# B08-01 — Random 선택과 머신러닝 기반 순차 선택 비교
## 실제 배양 데이터에 기반한 회고적 모의 실험

**연구 질문:** 동일한 초기 실측 조성과 후속 선택 예산을 사용할 때 Random 탐색과 Random Forest의 예측 최대값을 선택하는 탐색 방식 중 어느 쪽이 높은 **기존 실측 목표값**을 더 빨리 발견하는가?

- **자료:** B08-00 Clean v2가 만든 1,025개 실제 배지 조성, 44개 배지 성분. 이 중 목표값이 있는 조성만 사용.
- **실험 단위:** `medium_profile_id` (같은 조성의 반복 곡선을 각각 독립 후보로 사용하지 않음).
- **목표:** `K_median`, `r_median`을 **별개 실험으로** 최대화. 두 지표의 정확한 생물학적 정의는 아직 확인되지 않음.
- **방법:** 초기 관측값 30개를 공개한 뒤, 미공개 후보에서 매 단계 1개를 선택·기존 측정값을 공개하는 **오프라인 회고적 시뮬레이션**.
- **비교:** Random 선택 vs Random Forest 예측 최대값 선택(탐색성이 약한 순수 greedy 방식).
- **공정성:** 시드별 공통 초기 집합, 동일 예산, 재현 가능한 난수 및 동일 후보군.

> **주의:** 전체 데이터 중 미래 실험에 해당하는 실제값을 모델의 선택에 사용하지 않습니다. 단, 후보 집합 구성(목표값이 있는 조성의 선택)은 사후적으로 알고 있으므로 이 결과는 역사적 자료에 조건부인 *회고적 벤치마크*이며 실제 실험 배치·비용·외삽 성능 검증이 아닙니다.


In [ ]:
from pathlib import Path
import json, hashlib, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

cwd=Path.cwd().resolve()
ROOT=next((p for p in [cwd,*cwd.parents] if (p/'results'/'bioculture'/'B08_00_candidate_design_v2').is_dir()),None)
if ROOT is None:
    raise FileNotFoundError('B08-00 Clean v2 결과가 있는 bioimage-ai 프로젝트 루트에서 실행해 주세요.')
BASE=ROOT/'results'/'bioculture'
CAND=BASE/'B08_00_candidate_design_v2'/'candidate_pool_measured_profiles.csv'
FEATURE=BASE/'B07_00_ml_data_audit_v2'/'medium_feature_audit.csv'
OUT=BASE/'B08_01_sequential_selection_v1'
OUT.mkdir(parents=True,exist_ok=True)
for p in [CAND,FEATURE]:
    if not p.is_file(): raise FileNotFoundError(p)
SEEDS=[17,43,101]           # 독립적인 초기화 시나리오 (같은 관측 자료를 재사용)
N_INITIAL=30
N_STEPS=20                  # 추가 공개 횟수, 각 단계 1개 조성
N_TREES=100
print('Python',sys.version.split()[0], 'output',OUT)


## 1. 입력 감사 및 미래 실측값 격리

B08-00에서 이미 검증된 조성별 자료를 그대로 읽습니다. 입력 특성은 이름으로 명시된 44개 농도 열만 사용합니다. `K_median`, `r_median`, 라운드 식별자, 조건 ID, 순위 및 품질 열은 모델 입력에 포함되지 않습니다.


In [ ]:
df=pd.read_csv(CAND,encoding='utf-8-sig',dtype={'medium_profile_id':'string'},low_memory=False)
feature_info=pd.read_csv(FEATURE,encoding='utf-8-sig')
features=feature_info['feature'].astype(str).tolist()
TARGETS=['K_median','r_median']
assert len(df)==1025 and df.medium_profile_id.is_unique and len(features)==44
assert len(set(features))==44
assert set(features).issubset(df.columns)
assert set(TARGETS).issubset(df.columns)
assert not set(features).intersection(set(TARGETS)|{'medium_profile_id','round_membership','n_rounds','n_curve_records'})
assert all(pd.to_numeric(df[c],errors='coerce').notna().all() for c in features)
assert (df[features].apply(pd.to_numeric)<0).sum().sum()==0
print('조성:',len(df),'입력 성분:',len(features))
print('유효 목표별 후보:',{t:int(pd.to_numeric(df[t],errors='coerce').notna().sum()) for t in TARGETS})
print('초기 공개:',N_INITIAL,'후속 선택 횟수:',N_STEPS,'반복 시드:',SEEDS)


## 2. 선택 알고리즘과 실측값 공개 함수

미공개 실측값은 `_oracle`이라는 개별 객체에 보관합니다. 선택 전략에는 **미공개 후보의 조성 X와 공개된 (X, y) 쌍만** 전달합니다. 다음 코드에서 평가 함수는 반드시 선택이 확정된 후에 호출합니다. 

Random Forest는 예측 평균 최대값을 고르는 **탐욕적(greedy) 선택**입니다. 이번 실습은 Bayesian Optimization이 아니며, 예측 불확실성에 의한 탐색(exploration)도 적극 활용하지 않습니다.


In [ ]:
class HiddenOracle:
    def __init__(self, ids, values):
        self.__values=dict(zip([str(s) for s in ids],np.asarray(values,dtype=float)))
        self.revealed=set()
    def reveal(self, uid):
        uid=str(uid)
        if uid in self.revealed:
            raise AssertionError('동일 조성을 두 번 측정하도록 선택했습니다.')
        v=float(self.__values[uid])
        self.revealed.add(uid)
        return v
    def _audit_values_after_run(self):
        # 오직 결과 감사 단계에서만 사용; 선택 함수로 전달하지 않는다.
        return np.array(list(self.__values.values()),dtype=float)

def make_rf(seed):
    return Pipeline([
        ('imputer',SimpleImputer(strategy='median')),
        ('rf',RandomForestRegressor(n_estimators=N_TREES,max_features=0.8,
                                   min_samples_leaf=2,n_jobs=-1,random_state=int(seed)))
    ])

def choose_next(strategy, rng, observed_X, observed_y, unknown_X, unknown_ids, seed):
    if strategy=='Random':
        j=int(rng.integers(0,len(unknown_ids)))
        return str(unknown_ids[j]),np.nan
    if strategy=='RF_greedy':
        model=make_rf(seed)
        model.fit(observed_X,observed_y)
        score=model.predict(unknown_X)
        # 동점일 때 ID 순서는 바꾸지 않으므로 결과가 재현된다.
        j=int(np.argmax(score))
        return str(unknown_ids[j]),float(score[j])
    raise ValueError(strategy)

# 실제값을 '전체 회수하는 실수' 없이 선택 함수만으로 동작하는 간단한 함수 테스트
rng=np.random.default_rng(5)
assert choose_next('Random',rng,np.zeros((3,2)),np.arange(3),np.ones((4,2)),np.array(['a','b','c','d']),5)[0] in 'abcd'
print('선택 함수 구조 검사 통과')


## 3. 같은 초기 조건의 순차 실험

각 시드에서 전체 후보를 한 번 무작위로 섞어 초기 공개 30개를 구성합니다. 두 전략은 **동일한 초기 공개 집합**으로 출발합니다. 그 후 매 단계마다 미공개 후보를 1개 선택하고 공개된 데이터를 추가하여 다음 단계에 반영합니다.

기록: 각 단계 선택 ID, 공개값, 현재까지 최고 발견값(`best_so_far`), 모델의 선택 직전 예측값(`predicted_before_reveal`).


In [ ]:
def simulate_target(table, target, seed, strategy):
    sub=table.loc[table[target].notna(),['medium_profile_id',*features,target]].copy()
    sub=sub.sort_values('medium_profile_id').reset_index(drop=True)
    ids=sub.medium_profile_id.astype(str).to_numpy()
    X=sub[features].to_numpy(dtype=float)
    # 예측 전략은 이 벡터를 직접 전달받지 않는다: 오라클 내부로 캡슐화한다.
    oracle=HiddenOracle(ids,sub[target].to_numpy(dtype=float))
    rng0=np.random.default_rng(seed)
    perm=rng0.permutation(len(ids))
    initial_ix=perm[:N_INITIAL]
    seen=set()
    revealed=[]
    for j in initial_ix:
        uid=str(ids[j]); y=oracle.reveal(uid)
        seen.add(uid); revealed.append((uid,float(y)))
    best=max(y for _,y in revealed)
    initial_best=best
    trace=[]
    chooser_rng=np.random.default_rng(seed+100_000)
    for step in range(1,N_STEPS+1):
        known_mask=np.isin(ids,list(seen))
        unknown_mask=~known_mask
        known_ids=ids[known_mask]
        known_x=X[known_mask]
        y_lookup=dict(revealed) # 오직 공개된 값
        known_y=np.array([y_lookup[str(uid)] for uid in known_ids],dtype=float)
        unknown_x=X[unknown_mask]
        unknown_ids=ids[unknown_mask]
        assert len(unknown_ids)==len(ids)-N_INITIAL-step+1
        uid,pred=choose_next(strategy,chooser_rng,known_x,known_y,unknown_x,unknown_ids,seed+step)
        assert uid not in seen and uid in set(unknown_ids)
        # 중요: 선택 직후에야 숨긴 실측값을 단 하나 공개한다.
        y=oracle.reveal(uid)
        seen.add(uid);revealed.append((uid,y))
        best=max(best,y)
        trace.append({'target':target,'seed':seed,'strategy':strategy,'step':step,
                      'chosen_medium_profile_id':uid,'revealed_value':y,
                      'predicted_before_reveal':pred,'best_so_far':best,
                      'initial_best':initial_best,'n_revealed':N_INITIAL+step})
    assert len(seen)==N_INITIAL+N_STEPS and len(oracle.revealed)==len(seen)
    return trace

all_rows=[]
for target in TARGETS:
    for seed in SEEDS:
        for strategy in ['Random','RF_greedy']:
            rows=simulate_target(df,target,seed,strategy)
            all_rows.extend(rows)
            print(target,seed,strategy,'initial',round(rows[0]['initial_best'],4),
                  'final best',round(rows[-1]['best_so_far'],4))
traces=pd.DataFrame(all_rows)
assert len(traces)==len(TARGETS)*len(SEEDS)*2*N_STEPS
assert traces.groupby(['target','seed','strategy']).size().eq(N_STEPS).all()
assert traces.groupby(['target','seed'])['initial_best'].nunique().eq(1).all()
assert traces.groupby(['target','seed','strategy'])['chosen_medium_profile_id'].nunique().eq(N_STEPS).all()


## 4. 탐색 성과 비교

최종 최고 발견값과 초기 상태 대비 개선량을 비교합니다. `best_so_far`는 추가 공개에 따라 절대 감소하지 않아야 합니다. 전체 후보 중 최고 관측값은 결과 해석에만 쓰며, 후보 선택에 쓰지 않습니다.

**주의:** 세 시드는 같은 역사적 자료를 재사용하므로 독립 실험 3회가 아닙니다.


In [ ]:
summary=(traces.sort_values('step').groupby(['target','seed','strategy'],as_index=False)
         .tail(1)[['target','seed','strategy','initial_best','best_so_far','n_revealed']]
         .rename(columns={'best_so_far':'final_best'}))
summary['gain_from_initial']=summary['final_best']-summary['initial_best']
assert (summary.gain_from_initial>=-1e-12).all()
for key,g in traces.groupby(['target','seed','strategy']):
    assert np.all(np.diff(g.sort_values('step').best_so_far.to_numpy())>=-1e-12),key

# 누적 최적값 곡선에 대한 탐색적 면적 지표 (값이 클수록 초기에 높은 값을 발견)
aulc=(traces.groupby(['target','seed','strategy'],as_index=False)
      .agg(mean_best_over_steps=('best_so_far','mean')))
summary=summary.merge(aulc,on=['target','seed','strategy'],validate='one_to_one')
print('\n시드별 요약:')
print(summary.to_string(index=False))
print('\n전략별 요약(동일 역사적 자료의 시드 반복):')
print(summary.groupby(['target','strategy'])[['final_best','gain_from_initial','mean_best_over_steps']].agg(['mean','std']).round(4).to_string())


## 5. 성장곡선 그림 및 비교 설명

각 목표에 대해 전략별 최고 발견값의 단계별 평균을 표시합니다. 평균은 여러 개의 새 실험을 수행해 얻은 추정이 아니라 **초기 공개 시드에 따른 민감도 요약**입니다.


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4.2),constrained_layout=True)
for ax,t in zip(axes,TARGETS):
    s=traces[traces.target.eq(t)]
    for strategy in ['Random','RF_greedy']:
        g=s[s.strategy.eq(strategy)].groupby('step')['best_so_far']
        avg=g.mean(); lo=g.min(); hi=g.max()
        ax.plot(avg.index,avg.to_numpy(),label=strategy)
        ax.fill_between(avg.index,lo.to_numpy(),hi.to_numpy(),alpha=.13)
    ax.set_title(f'{t}: best observed so far')
    ax.set_xlabel('Additional revealed profiles')
    ax.set_ylabel('Best revealed outcome')
    ax.legend()
path_fig=OUT/'sequential_best_so_far.png'
fig.savefig(path_fig,dpi=155)
plt.show()


## 6. 결과 저장·파일 해시·재읽기 검사

저장되는 행은 모두 **실제로 선택되어 공개된 조성**입니다. 전 후보의 숨겨진 목표값을 담은 예측표를 생성하지 않습니다. 결과는 `results/bioculture/B08_01_sequential_selection_v1/`에 저장합니다.


In [ ]:
def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''):h.update(block)
    return h.hexdigest()

p_trace=OUT/'sequential_selection_trace.csv'
p_summary=OUT/'sequential_selection_summary.csv'
p_config=OUT/'B08_01_validation_report.json'
traces.to_csv(p_trace,index=False,encoding='utf-8-sig')
summary.to_csv(p_summary,index=False,encoding='utf-8-sig')
report={'status':'RETROSPECTIVE_OFFLINE_SIMULATION_NOT_NEW_EXPERIMENT',
        'n_all_profiles':int(len(df)),'n_features':len(features),
        'valid_profile_counts':{t:int(df[t].notna().sum()) for t in TARGETS},
        'targets':TARGETS,'seeds':SEEDS,'n_initial':N_INITIAL,'n_additional_steps':N_STEPS,
        'strategies':['Random','RF_greedy'],'oracle_isolation':'algorithm receives only revealed target values',
        'claims_not_supported':['new biological experiments','causal nutrient effects',
            'unbiased external generalization','statistically calibrated uncertainty'],
        'target_definitions_verified':False}
p_config.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
read_trace=pd.read_csv(p_trace,encoding='utf-8-sig')
read_summary=pd.read_csv(p_summary,encoding='utf-8-sig')
assert len(read_trace)==len(traces) and len(read_summary)==len(summary)
assert read_trace.groupby(['target','seed','strategy']).size().eq(N_STEPS).all()
manifest=pd.DataFrame([{'path':str(p.relative_to(ROOT)),'bytes':p.stat().st_size,'sha256':sha256(p),
                        'role':'input' if p in [CAND,FEATURE] else 'output'}
                       for p in [CAND,FEATURE,p_trace,p_summary,p_config,path_fig]])
manifest.to_csv(OUT/'input_output_sha256_manifest.csv',index=False,encoding='utf-8-sig')
print('검증 통과:',report['status'])
print('Saved:',*[p.name for p in [p_trace,p_summary,p_config,path_fig]],sep='\n- ')


## 7. 해석과 과학적 한계

1. `K_median`, `r_median`은 기존 평가표에서 온 지표의 조성별 중앙값입니다. 정의·실험적 정밀도에 관한 검증을 이 실습만으로 완료하지 않았습니다.
2. 초기 선택 집합을 제외한 목표값이 **완전하게 숨겨진 것은 코드 구조 안에서의 접근 권한**에 관한 사항입니다. 연구자는 사후적으로 수집된 기록을 활용하므로 회고적 선택 편향이 남습니다.
3. 본 실습은 동일 데이터셋 내 순차적 재공개입니다. **새 실험이나 최적 배지의 실제 검증이 아닙니다.**
4. 동일한 배지 조성을 여러 번 선택하지 않으며, 라운드 교란·배지 조성의 조합 분포 이동·실험 비용은 아직 명시적으로 최적화하지 않았습니다.
5. RF greedy는 탐색이 약합니다. 다음 B08-02의 Bayesian Optimization에서는 불확실성 기반 탐색을 비교합니다.

### 연습문제 (해답과 실행 가능한 Python 풀이는 별도 Word 해설서에 수록)

1. 초기 공개 수를 30에서 50으로 늘리면 공정한 비교를 위해 무엇을 고정해야 합니까?
2. `best_so_far`와 마지막으로 선택한 후보의 관측값은 왜 다른 수치입니까? 코드로 확인하십시오.
3. 같은 시드·목표에서 두 전략의 초기 최고값이 반드시 같은지 검증하십시오.
4. Random과 RF 탐욕적 선택의 목표별 최종 최고 발견값을 비교하고, 차이가 일반화 능력의 증명인지 설명하십시오.
5. `medium_profile_id`가 중복 선택되지 않는지 검증하는 코드를 작성하십시오.
